**Explanation:** This cell installs or upgrades the libraries required for Unsloth fine-tuning, Transformers, TRL, and Accelerate.


In [1]:
!pip install --upgrade unsloth transformers trl==0.12.0 accelerate wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 853.5 kB/s eta 0:00:00 0:00:01
INFO: pip is looking at multiple versions of unsloth to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.0 MB/s eta 0:00:00
INFO: pip is still looking at multiple versions of unsloth to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 k

**Explanation:** This cell logs in to Hugging Face using a direct access token so the notebook can access gated models or private repositories.


In [ ]:
from huggingface_hub import login

hf_token = "hf_token"  
login(token=hf_token)
print("Hugging Face login completed.")


Hugging Face login completed.


In [ ]:
import wandb
wandb.login(key="wandb_key")  # Replace with your actual WandB API key

wandb.init(project="llama3-egyptian-legal-chatbot", name="llama3-8b-lora-run1")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amotaz792 (amotaz792-the-egyptian-e-learning-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


**Explanation:** This cell converts a `.txt` file into the Llama 3 conversation format expected by the fine-tuning dataset.


In [ ]:
from pathlib import Path

input_file = Path("/kaggle/input/datasets/ahmedelshiekh792/fine-tuning/egyptian_laws.txt")
output_file = Path("llama3_formatted_data.txt")
default_user_prompt = "اكتب ردًا مفيدًا وواضحًا باللهجة المصرية اعتمادًا على النص المتاح."

if input_file.suffix.lower() != ".txt":
    raise ValueError("The input file must be a .txt file.")

full_text = input_file.read_text(encoding="utf-8")
chunks = [chunk.strip() for chunk in full_text.split("\n\n") if chunk.strip()]

if len(chunks) <= 1:
    chunks = [line.strip() for line in full_text.splitlines() if line.strip()]


def format_txt_chunk(chunk):
    return (
        "<|begin_of_text|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{default_user_prompt}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n{chunk}<|eot_id|>"
    )

samples = [format_txt_chunk(chunk) for chunk in chunks]

with output_file.open("w", encoding="utf-8") as outfile:
    for sample in samples:
        outfile.write(sample + "\n")

print(f"Formatted {len(samples)} text samples and saved them to {output_file}")


**Explanation:** This cell loads the formatted Kaggle training file, converts it into individual Llama 3 conversation samples, and creates the training dataset.


In [ ]:
from datasets import Dataset

file_path = "llama3_formatted_data.txt"

with open(file_path, "r", encoding="utf-8") as f:
    conversations = [line.strip() for line in f if line.strip()]

dataset = Dataset.from_list([{"text": conv} for conv in conversations])

print(f"Total samples for training: {len(dataset)}")
print("\n--- First 2 samples ---")
for i in range(min(2, len(dataset))):
    print(f"Sample {i}:\n{dataset[i]['text']}\n")


Total samples for training: 11177

--- First 2 samples ---
Sample 0:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Sample 1:
إيه هي أبرز أنواع الأدلة اللي ممكن تستخدم عشان نثبت إن فيه جريمة حصلت؟<|eot_id|><|start_header_id|>assistant<|end_header_id|>



**Explanation:** This cell loads the Llama 3.1 8B Instruct model and tokenizer with 4-bit quantization to reduce GPU memory usage.


In [6]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


==((====))==  Unsloth 2025.10.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

**Explanation:** This cell adds LoRA adapters so fine-tuning updates a small set of trainable adapter weights instead of the full model.


In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.10.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


**Explanation:** This cell configures the supervised fine-tuning trainer with the dataset, tokenizer, data collator, and training hyperparameters.


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # مخلينها False عشان الأمان والتوافق الكامل
    args = TrainingArguments(
        per_device_train_batch_size = 2, # قللناها لـ 2 عشان كارت الشاشة ميهنجش ويقف في النص
        gradient_accumulation_steps = 8, # رفعناها لـ 8 عشان نعوض التقليل ونحافظ على جودة التدريب
        warmup_steps = 5,
        num_train_epochs = 1, 
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1, 
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb", 
        average_tokens_across_devices = False, # السطر السحري بتاع حل كراش الـ mean
    ),
)

/kaggle/working/unsloth_compiled_cache/UnslothSFTTrainer.py:690: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/kaggle/working/unsloth_compiled_cache/UnslothSFTTrainer.py:704: UserWarning: You passed a `dataset_num_proc` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/kaggle/working/unsloth_compiled_cache/UnslothSFTTrainer.py:718: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map (num_proc=2):   0%|          | 0/11177 [00:00<?, ? examples/s]

**Explanation:** This cell starts the fine-tuning run and stores the training statistics returned by the trainer.


In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,177 | Num Epochs = 1 | Total steps = 350
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.391400
2,3.383400
3,3.146100
4,2.947400
5,3.037900
6,3.054200
7,2.739600
8,2.749100
9,2.418300
10,2.466400


**Explanation:** This cell saves the fine-tuned LoRA adapters and tokenizer to a local output folder.


In [ ]:
model.save_pretrained("Ahmed_Moataz_egyptian_chatbot")
tokenizer.save_pretrained("Ahmed_Moataz_egyptian_chatbot")
print("Model and tokenizer saved successfully.")


Model and tokenizer saved successfully.


In [11]:
# 1. الحفظ أونلاين على حسابك في Hugging Face مباشرة كموديل مدموج جاهز
model.push_to_hub_merged(
    "Ahmed792/lora_model", # اكتب هنا اسم الموديل اللي تحبه على حسابك
    tokenizer, 
    save_method = "merged_16bit", # أو "merged_4bit" لو عايزه خفيف وموفر مساحة
    token = hf_token
)

config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [05:45<01:54, 114.66s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [06:14<00:00, 93.65s/it] 


Unsloth: Merge process complete. Saved to `/kaggle/working/Ahmed792/lora_model`


**Explanation:** This cell applies the Llama 3 chat template, switches the model to inference mode, and runs a streamed Egyptian Arabic test prompt.


In [12]:
from transformers import TextStreamer
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "إيه هي أبرز أنواع الأدلة اللي ممكن تستخدم عشان نثبت إن فيه جريمة حصلت؟"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.5,
    min_p = 0.1,
)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


أبرز أنواع الأدلة دي هي الأدلة المادية زي الأسلحة والأدوات المستخدمة في الجريمة، الأدلة الشفهية اللي بتسمعها الشهود، الأدلة الجنائية اللي بتكون في جثث الضحايا أو في أماكن الجريمة، والأدلة اللي بتكون في المستندات والمستندات الإلكترونية.<|eot_id|>


**Explanation:** This cell creates a simple interactive chat interface for testing the fine-tuned model directly in the notebook.


In [18]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from unsloth import FastLanguageModel
import torch

# 1. تفعيل وضع الـ Inference السريع
FastLanguageModel.for_inference(model)

# 2. تصميم واجهة مربع الإدخال والزرار
text_input = widgets.Text(
    value='',
    placeholder='اكتب رسالتك هنا بالمصري...',
    description='الرسالة:',
    disabled=False,
    layout=widgets.Layout(width='70%')
)

button = widgets.Button(
    description='إرسال 🚀',
    disabled=False,
    button_style='success',
    tooltip='اضغط للتحدث مع البوت',
    icon='paper-plane'
)

output_area = widgets.Output()

# 3. دالة تشغيل الموديل عند الضغط على الزرار
def on_button_clicked(b):
    user_message = text_input.value.strip()
    if not user_message:
        return
        
    with output_area:
        clear_output() # مسح الرد القديم
        print(f"أنت: {user_message}\n")
        print("البوت: جاري التفكير... 🤔", end="\r")
        
        # تجهيز التمبليت
        messages = [{"role": "user", "content": user_message}]
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")
        
        # توليد الرد في الخلفية بدون Streamer لتجنب مشاكل الـ Printing البصرية
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            use_cache=True,
            temperature=0.5,
            min_p=0.1,
            eos_token_id=tokenizer.eos_token_id
        )
        
        # قص الـ Prompt وفك التشفير عن الإجابة فقط
        generated_tokens = outputs[0][len(inputs[0]):]
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        # تنظيف نهائي وأمان تام من أي توكنز عنيدة
        response = response.replace("<|eot_id|>", "").strip()
        
        # مسح جملة "جاري التفكير" وطباعة الرد النظيف
        clear_output(wait=True)
        print(f"أنت: {user_message}\n")
        print(f"البوت: {response}")
        print("\n" + "_"*50 + "\n")

# ربط الزرار بالدالة
button.on_click(on_button_clicked)

# 4. عرض المربع والزرار على الشاشة
print("💬 شات بوت مصري تفاعلي جاهز للكتابة:")
display(widgets.HBox([text_input, button]))
display(output_area)

💬 شات بوت مصري تفاعلي جاهز للكتابة:


Output()